In [1]:
def get_weather( city: str, unit: str ):
    """
    Retrives the weather for a city in the specified unit.
    
    :param city: The city name.
    :type city: str
    :param unit: the temprature unit, either'Celsius' or 'fahrenheit'.
    :type unit: str
    """

    return {"status": "success", "report": f"Weather for {city} is sunny."}


In [ ]:
# optional params :- with default values 

def search_flights(destination: str, departure_date: str, flexible_days: int = 0):
    """
    Searches for flights.

    Args:
        destination (str): The destination city.
        departure_date (str): The desired departure date.
        flexible_days (int, optional): Number of flexible days for the search. Defaults to 0.
    """
    # function logic
    if flexible_days > 0:
        return {"status": "success", "report": f"Found flexible flights to {destination}."}
    return {"status": "success", "report": f"Found flights to {destination} on {departure_date}."}

In [ ]:
# Optional Parameters with typing.Optional
from typing import Optional

def create_user_profile(username: str, bio: Optional[str] = None):
    """
    Creates a new user profile.

    Args:
        username (str): The user's unique username.
        bio (str, optional): A short biography for the user. Defaults to None.
    """
    # function logic 
    if bio:
        return {"status": "success", "message": f"Profile for {username} created with a bio."}
    return {"status": "success", "message": f"Profile for {username} created."}


In [ ]:
# All tool calls within a single agent turn share the same InvocationContext. 
# This means they also share the same temporary (temp:) state, which is how data can be passed between them.
"""
Best Practices¶
While you have considerable flexibility in defining your function, remember that simplicity enhances usability for the LLM. Consider these guidelines:

Fewer Parameters are Better: Minimize the number of parameters to reduce complexity.
Simple Data Types: Favor primitive data types like str and int over custom classes whenever possible.
Meaningful Names: The function's name and parameter names significantly influence how the LLM interprets and utilizes the tool. 
Choose names that clearly reflect the function's purpose and the meaning of its inputs. Avoid generic names like do_stuff() or beAgent().
Build for Parallel Execution: Improve function calling performance when multiple tools are run by building for asynchronous operation. For information on enabling parallel execution for tools, see Increase tool performance with parallel execution."""

In [ ]:
# Long Running Function Tools
"""Depending on the type of tool you are building, designing for asynchronous operation may be a better solution than creating a long running tool. 
For more information, see Increase tool performance with parallel execution."""

In [ ]:
# 1. Define the long running function
from typing import Any
# from google.generativeai import f/
from google.adk.tools import LongRunningFunctionTool

def ask_for_approval(
    purpose: str, amount: float
) -> dict[str, Any]:
    """Ask for approval for the reimbursement."""
    # create a ticket for the approval
    # Send a notification to the approver with the link of the ticket
    return {'status': 'pending', 'approver': 'Sean Zhou', 'purpose' : purpose, 'amount': amount, 'ticket-id': 'approval-ticket-1'}

def reimburse(purpose: str, amount: float) -> str:
    """Reimburse the amount of money to the employee."""
    # send the reimbrusement request to payment vendor
    return {'status': 'ok'}

# 2. Wrap the function with LongRunningFunctionTool
long_running_tool = LongRunningFunctionTool(func=ask_for_approval)

In [ ]:
"""When using a LongRunningFunctionTool, your function can initiate the long-running operation and optionally return an initial result, such as a long-running operation id.
 Once a long running function tool is invoked the agent runner pauses the agent run and lets the agent client to decide whether to continue or wait until the long-running operation finishes. 
 The agent client can query the progress of the long-running operation and send back an intermediate or final response.
 The agent can then continue with other tasks. An example is the human-in-the-loop scenario where the agent needs human approval before proceeding with a task."""

In [ ]:
# Agent Interaction
# from google.adk.tools import Event
# from google.adk.tools import FunctionCsl

# Each event may have multiple parts:
# text,function_call,function_response

from typing import types
import asyncio
from typing import Optional
from google.genai import types
from google.genai.types import Event

# If you are using Long Running tools
from google.genai.types import FunctionCall, FunctionResponse
from google.adk.runners import Runner
from google.adk.sessions import Session


from google.adk.agents import Agent

MODEL_NAME = "gemini-2.0-flash"
USER_ID = "user-1"

# AGENT SETUP 
def create_agent() -> Agent:
    return Agent(
        name="approval-agent",
        model=MODEL_NAME,
        instructions=(
            "You are an assistant that can request approvals "
            "using long-running tools and continue after approval."
        ),
    )

#  SESSION + RUNNER 
async def setup_session_and_runner():
    agent = create_agent()

    session = Session( agent=agent,user_id=USER_ID,)

    runner = Runner(agent=agent)
    return session, runner



async def call_agent_async(query):

    def get_long_running_function_call(event: Event) -> types.FunctionCall:
        # Get the long running function call from the event

        if not event.long_running_tool_ids or not event.content or not event.content.parts:
            return
        
        for part in event.content.parts:
            if (part and part.function_call and event.long_running_tool_ids
                and part.function_call.id in event.long_running_tool_ids
            ):
                return part.function_call

    def get_function_response(event: Event, function_call_id: str) -> types.FunctionResponse:
        # Get the function response for the fuction call with specified id.
        if not event.content or not event.content.parts:
            return
        for part in event.content.parts:
            if (
                part
                and part.function_response
                and part.function_response.id == function_call_id
            ):
                return part.function_response

    # create user message content
    content = types.Content(role='user', parts=[types.Part(text=query)])
    session, runner = await setup_session_and_runner()

    print("\nRunning agent...")
    events_async = runner.run_async(
        session_id=session.id, user_id=USER_ID, new_message=content
    )

    #defining the state content 
    long_running_function_call, long_running_function_response, ticket_id = None, None, None
    async for event in events_async:
        # Use helper to check for the specific auth request event
        if not long_running_function_call:
            long_running_function_call = get_long_running_function_call(event)
        else:
            _potential_response = get_function_response(event, long_running_function_call.id)
            if _potential_response: # Only update if we get a non-None response
                long_running_function_response = _potential_response
                ticket_id = long_running_function_response.response['ticket-id']
        if event.content and event.content.parts:
            if text := ''.join(part.text or '' for part in event.content.parts):
                print(f'[{event.author}]: {text}')


    if long_running_function_response:
        # query the status of the correpsonding ticket via tciket_id
        # send back an intermediate / final response
        updated_response = long_running_function_response.model_copy(deep=True)
        updated_response.response = {'status': 'approved'}
        async for event in runner.run_async(
          session_id=session.id, user_id=USER_ID, new_message=types.Content(parts=[types.Part(function_response = updated_response)], role='user')
        ):
            if event.content and event.content.parts:
                if text := ''.join(part.text or '' for part in event.content.parts):
                    print(f'[{event.author}]: {text}')

In [ ]:
"""
User query
   ↓
Agent starts reasoning
   ↓
Agent calls long-running tool
   ↓
Tool returns "pending"
   ↓
External system approves
   ↓
Updated response sent back
   ↓
Agent continues + responds
"""